In [ ]:
import pandas as pd 

df = pd.read_csv("/workspace/kor_med_opendataset/snuh_ClinicalQA/train.csv")

row_sample = df.head(1).to_dict(orient="records")[0]
row_sample

import ast
def get_snuh_ClinicalQA_prompt(row):
    # Parse options safely (assuming it's a string representation of a dict)

    # options 파싱 및 포맷팅
    options_dict = ast.literal_eval(row['options'])
    formatted_options = "\n".join([
        f"{key.replace('option_', '').upper()}) {value}"
        for key, value in sorted(options_dict.items())
    ])

    # Qwen에 최적화된 한국어 프롬프트
    snuh_ClinicalQA_prompt = f"""당신은 숙련된 내과 전문의입니다. 다음 임상 사례를 바탕으로 환자의 주소증에 대한 **가장 가능성 높은 단일 원인**을 판단하세요.

    제시된 보기 중에서 정답을 선택하고, 반드시 **유효한 JSON 형식**으로만 응답하세요.  
    - "answer" 필드는 "**X) 보기 텍스트**" 형식을 정확히 따르세요 (예: "C) 원발성 담즙성 담관염").  
    - "explanation" 필드는 근거를 설명하세요.  
    - JSON 외의 어떤 추가 텍스트도 출력하지 마세요.

    질문: {row['question']}

    보기:
    {formatted_options}
    """

    return snuh_ClinicalQA_prompt

{'question_id': 1,
 'chief_complaint': '가려움증',
 'purpose': '가려움증을 호소하는 사람에게서 병력, 신체소견, 피부소견, 검사소견을 종합하여 원인을 감별할 수 있다.',
 'question': '62세 남자 환자가 3개월 전부터 시작된 전신 가려움증을 주소로 내원하였다. 환자는 특별한 피부 발진은 없으나 밤에 특히 가려움이 심해져 수면에 방해를 받는다고 호소한다. 최근 6개월 동안 체중이 5kg 감소했으며 쉽게 피로해진다고 한다. 과거력상 고혈압으로 약물 복용 중이며, 1년 전부터 혈당이 약간 높아 식이조절 중이다. 신체검진에서 피부는 건조하고 긁힌 자국이 등과 사지에 관찰되며, 우상복부에 압통 없이 촉진되는 간비대(2cm)가 있다. 이 환자의 가려움증 원인으로 가장 가능성이 높은 것은?',
 'exam': '혈액검사: WBC 5,500/μL, Hb 11.2g/dL, Platelet 240,000/μL, AST 85U/L, ALT 92U/L, ALP 420U/L, γ-GT 380U/L, 총 빌리루빈 2.8mg/dL, 직접 빌리루빈 1.9mg/dL, BUN 18mg/dL, Cr 0.9mg/dL, Na 138mEq/L, K 4.1mEq/L, 공복혈당 130mg/dL',
 'options': "{'option_A': '당뇨병', 'option_B': '약물 부작용', 'option_C': '원발성 담즙성 담관염', 'option_D': '만성 신부전', 'option_E': '건조피부증(노인성 소양증)'}",
 'answer': 'C',
 'explanation': '이 환자는 전신 가려움증, 체중 감소, 피로감을 호소하며 검사실 소견에서 담즙정체성 간기능 이상(AST, ALT의 경도 상승과 ALP, γ-GT의 현저한 상승, 빌리루빈 상승)이 관찰됩니다. 이러한 소견은 원발성 담즙성 담관염(Primary Biliary Cholangitis, 과거 Primary Biliary Cirrhosis)을 강하게 시사합니다. 원발성

## 

## sean0042_KorMedMCQA

In [ ]:
import pandas as pd
from glob import glob 
# 데이터 합치기 
for domain in ['doctor', 'nurse', 'dentist', 'pharm']:
    dev = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_dev.csv")
    fewshot = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_fewshot.csv")
    train = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_train.csv")
    test = pd.read_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_test.csv")
    all = pd.concat([dev, fewshot, train, test])
    # 중복치 제거
    all = all.drop_duplicates(subset=['question'])
    # all.to_csv(f"/workspace/kor_med_opendataset/sean0042_KorMedMCQA/{domain}/{domain}_all.csv", index=False)

In [2]:
import sys 
sys.path.append("/workspace")
import pandas as pd 
from src.qa_prompt import get_sean0042_KorMedMCQA_prompt
# prompt 만들기
df = pd.read_csv("/workspace/kor_med_opendataset/sean0042_KorMedMCQA/doctor/doctor_all.csv")


row = df.head(1).to_dict(orient="records")[0]

prompt = get_sean0042_KorMedMCQA_prompt(row)

print(prompt)

당신은 의사입니다.
질문:
광역시 소재 대학병원에 소속된 내과 전문의 A가 콜레라 환자를 진단했다. A가 할 조치는?

보기:
1) 병원장에게 보고
2) 광역시장에게 신고
3) 질병관리청장에게 신고
4) 관할 보건소장에게 신고
5) 보건복지부장관에게 신고

질문을 분석하고, 제시된 보기 중에서 한개의 보기를 선택하고, 근거를 설명해.
- JSON 외의 어떤 추가 텍스트도 출력하지 마세요.- 정답을 다음 JSON 형식으로만 답하세요:
{"answer":"보기","explanation":"한국어 근거"}
